# Lab 2.2: FlashAttention Implementation

## Objective
Implement FlashAttention-2 from scratch, understanding tiling, online softmax, and IO-aware computation.

## Learning Goals
1. Understand the FlashAttention algorithm and its improvements over standard attention
2. Implement tiling for Q, K, V matrices to reduce HBM accesses
3. Implement online softmax for numerical stability
4. Combine tiling and online softmax into a forward pass
5. Benchmark against PyTorch attention and analyze performance

## Prerequisites
- Completion of Lab 2.1 (KV caching)
- Basic understanding of GPU memory hierarchy (HBM vs SRAM)
- Familiarity with PyTorch and CUDA (optional)

## Modern Context
FlashAttention-2 (2023) improves parallelism and work partitioning, achieving up to 2× speedup over FlashAttention. This lab guides you through implementing the core algorithm.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time
import matplotlib.pyplot as plt
from typing import Tuple, Optional

plt.style.use('seaborn-v0_8')
%matplotlib inline

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

## Part 1: Understanding FlashAttention

FlashAttention reduces memory reads/writes by tiling and using online softmax. The key idea: compute attention block by block, keeping a running maximum and sum.

### Algorithm Overview
1. Split Q, K, V into tiles that fit in SRAM
2. For each tile of Q, compute attention scores with tiles of K, V
3. Update output and normalization statistics incrementally
4. Write final output after processing all tiles

### Memory Hierarchy
- **HBM**: High bandwidth memory (slow, large)
- **SRAM**: On-chip memory (fast, small)
- **Goal**: Minimize HBM accesses by reusing data in SRAM

In [ ]:
def compute_tile_attention(Q_tile: torch.Tensor, K_tile: torch.Tensor, V_tile: torch.Tensor) -> torch.Tensor:
    """
    Compute attention for a single tile.
    
    Args:
        Q_tile: [tile_size, d]
        K_tile: [tile_size, d]
        V_tile: [tile_size, d]
    
    Returns:
        O_tile: [tile_size, d]
    """
    # Compute attention scores for this tile
    d = Q_tile.shape[-1]
    S = Q_tile @ K_tile.T / (d ** 0.5)
    P = F.softmax(S, dim=-1)
    O = P @ V_tile
    return O

def split_into_tiles(tensor: torch.Tensor, tile_size: int) -> list:
    """
    Split tensor into tiles along the sequence dimension.
    
    Args:
        tensor: [n, d]
        tile_size: size of each tile
    
    Returns:
        List of tiles [tile_size, d] (last tile may be smaller)
    """
    n = tensor.shape[0]
    tiles = []
    for start in range(0, n, tile_size):
        end = min(start + tile_size, n)
        tile = tensor[start:end]
        tiles.append(tile)
    return tiles


## Part 2: Online Softmax

Standard softmax requires seeing all scores to compute the maximum. Online softmax computes running max and sum, allowing incremental updates.

We maintain:
- m: running maximum
- l: running sum of exponentials (scaled)
- o: running output

In [ ]:
def online_softmax_update(new_scores: torch.Tensor, V_tile: torch.Tensor, prev_m: torch.Tensor, prev_l: torch.Tensor, prev_o: torch.Tensor) -> Tuple[torch.Tensor, torch.Tensor, torch.Tensor]:
    """
    Update online softmax statistics with new scores and value tile.
    
    Args:
        new_scores: [tile_size, tile_size] or [tile_size, n]
        V_tile: [tile_size, d] value tile
        prev_m: previous running max [tile_size]
        prev_l: previous running sum [tile_size]
        prev_o: previous running output [tile_size, d]
    
    Returns:
        updated m, l, o
    """
    # Compute local max over new scores (columns)
    m_new = new_scores.max(dim=-1).values
    m_new = torch.maximum(prev_m, m_new)
    
    # Rescale previous accumulators
    scale_prev = torch.exp(prev_m - m_new)
    l_new = scale_prev * prev_l
    o_new = scale_prev.unsqueeze(-1) * prev_o
    
    # Compute exponentials of new scores (with proper scaling)
    scores_scaled = new_scores - m_new.unsqueeze(-1)
    exp_scores = torch.exp(scores_scaled)
    
    # Update l and o with new scores
    l_new = l_new + exp_scores.sum(dim=-1)
    o_new = o_new + exp_scores @ V_tile
    
    return m_new, l_new, o_new


## Part 3: FlashAttention Forward Pass

Combine tiling and online softmax to implement the full forward pass.

In [ ]:
def flash_attention_forward(Q: torch.Tensor, K: torch.Tensor, V: torch.Tensor, tile_size: int = 64) -> torch.Tensor:
    """
    FlashAttention forward pass.
    
    Args:
        Q, K, V: [batch, heads, seq_len, d_head]
        tile_size: size of tiles for Q and K/V
    
    Returns:
        O: [batch, heads, seq_len, d_head]
    """


## Part 4: Benchmarking

Compare your implementation with PyTorch's standard attention and measure speedup.

In [ ]:
def benchmark_attention(seq_len: int, d_head: int = 64, num_heads: int = 8, batch_size: int = 2):
    """
    Benchmark different attention implementations.
    """
    # Generate random data
    Q = torch.randn(batch_size, num_heads, seq_len, d_head, device='cuda' if torch.cuda.is_available() else 'cpu')
    K = torch.randn(batch_size, num_heads, seq_len, d_head, device='cuda' if torch.cuda.is_available() else 'cpu')
    V = torch.randn(batch_size, num_heads, seq_len, d_head, device='cuda' if torch.cuda.is_available() else 'cpu')
    
    # Benchmark PyTorch attention
    start = time.time()
    # Use PyTorch scaled_dot_product_attention or manual matmul
    end = time.time()
    pytorch_time = end - start
    
    # Benchmark your FlashAttention
    start = time.time()
    # output = flash_attention_forward(Q, K, V, tile_size=64)
    end = time.time()
    flash_time = end - start
    
    print(f"Seq len {seq_len}: PyTorch {pytorch_time:.4f}s, FlashAttention {flash_time:.4f}s, speedup {pytorch_time/flash_time:.2f}x")
    return pytorch_time, flash_time

lengths = [64, 128, 256, 512, 1024]

times = [benchmark_attention(l) for l in lengths]
pytorch_times = [t[0] for t in times]
flash_times = [t[1] for t in times]
plt.figure(figsize=(10, 6))
plt.plot(lengths, pytorch_times, marker="o", label="PyTorch Attention")
plt.plot(lengths, flash_times, marker="s", label="FlashAttention (ours)")
plt.xlabel("Sequence Length")
plt.ylabel("Time (seconds)")
plt.title("Attention Implementation Benchmark")
plt.legend()
plt.grid(True)
plt.show()


## Conclusion

You've implemented FlashAttention from scratch, understanding the key optimizations that make it memory-efficient and fast.

### Further Exploration
1. Implement backward pass (FlashAttention also optimizes gradient computation)
2. Add support for dropout, masking, and different attention variants
3. Compare with existing implementations (xFormers, Triton kernels)
4. Profile HBM accesses using NVIDIA Nsight Systems

### Submission
Submit your completed notebook with implementations and benchmarking results.